In [86]:
from collections import defaultdict
import csv


def spec_check(ledger_file_path: str, goal_ledger_seq: int = 12):

    ledgers_data = defaultdict(list)
    try:
        with open(ledger_file_path) as csvfile:
            reader = csv.DictReader(csvfile)
            for row in reader:
                # Basic type conversion and validation
                if ("validated" in row["ledger_seq"]):
                    continue
                try:
                    node_id = int(row["peer_id"])
                    ledger_seq = int(row["ledger_seq"])
                    ledger_hash = row.get("ledger_hash", "").strip()
                    if ledger_hash:  # this checks that it's not empty or just spaces
                        parsed_row = {
                            "node_id": node_id,
                            "ledger_seq": ledger_seq,
                            "ledger_hash": ledger_hash,
                            "validated": row["validated"] == "True",
                        }
                        ledgers_data[ledger_seq].append(parsed_row)
                except (ValueError, KeyError) as e:
                    continue
    except csv.Error as e:
        raise IOError(f"Error reading ledger file: {ledger_file_path}") from e

    all_hashes_pass = True
    all_sequences_pass = True
    all_ledger_goal_reached = True

    for _, records in ledgers_data.items():
        first_validated_hash = next(
            (x["ledger_hash"] for x in records if x["validated"] and x["ledger_hash"] != "NOT FOUND"),
            None
        )

        ledger_hashes_same = all(
            x["ledger_hash"] == first_validated_hash
            for x in records
            if x["ledger_hash"] != "NOT FOUND" and x["validated"]
        )

        ledger_seq_same = all(
            x["ledger_seq"] == records[0]["ledger_seq"] for x in records if x["ledger_seq"] != -1
        )
        all_hashes_pass &= ledger_hashes_same
        all_sequences_pass &= ledger_seq_same


    all_ledger_goal_reached &= all(entry["validated"] for entry in ledgers_data[goal_ledger_seq])

    # all_ledger_goal_reached &= len(set(entry["ledger_hash"] for entry in ledgers_data[goal_ledger_seq])) == 1

    # print("Safety:", all_hashes_pass)
    # print("Liveness:", all_ledger_goal_reached)

    if not all_hashes_pass and all_ledger_goal_reached:
        print("djsdjdsj")

    return not all_hashes_pass

In [87]:
import os

import numpy as np

from scipy.stats import pearsonr, pointbiserialr, boschloo_exact
from statsmodels.tsa.stattools import adfuller

def build_identifier(label: str, gen: int, tc: int, retr: int):
    return f"{label}_G{gen}T{tc}R{retr}"

grand_result = {}

for run_label in [
    "AMG_Random",
    "AMG_NSGAII",
    "AMG_TournamentDCD",
    "AMG_TimeElitism",
    "AMG_ProposalElitism",
    "AMG_TimeTournament",
    "AMG_ProposalTournament",
    "AMG_TimeRoulette",
    "AMG_ProposalRoulette",
]:
    tdir = f"./logs/{run_label}"
    results = {}

    for gen in range(1, 51):
        results[gen] = {}

        for tc in range(1, 11):
            retr = 0
            encoding = ""

            with open(f"{tdir}/{build_identifier(run_label, gen, tc, retr)}/run_info.txt", "r") as file:
                lines = file.readlines()
                encoding = lines[1].strip()  # Remove newline character

            while True:
                filepath = f"{tdir}/{build_identifier(run_label, gen, tc, retr+1)}/run_info.txt"

                if os.path.exists(filepath):
                    with open(filepath, "r") as file:
                        lines = file.readlines()

                        if encoding == lines[1].strip():
                            retr += 1
                        else:
                            break
                else:
                    break

            
            import re

            filepath = f"{tdir}/{build_identifier(run_label, gen, tc, retr)}/run_info.txt"

            with open(filepath, "r") as file:
                lines = file.readlines()
                third_line = lines[2]
                fourth_line = lines[3]
                fifth_line = lines[4]

                # Extract validation time
                match = re.search(r"Average validation time:\s*([\d.]+)\s*seconds", third_line)
                validation_time = float(match.group(1)) if match else None

                # Extract proposal sequence
                match = re.search(r"Proposal count:\s*([\d.]+)", fourth_line)
                proposal_sequence = float(match.group(1)) if match else None

                # Extract total violations
                violation_match = re.search(r"Total violations:\s*(\d+)", fifth_line)
                total_violations = int(violation_match.group(1)) if violation_match else None

                # # Apply condition
                # if total_violations == 1:
                #     validation_time = 0

                # if validation_time is not None:
                #     print(f"Adjusted average validation time: {validation_time} seconds")
                # else:
                #     print("Validation time not found.")

            filename = f"{tdir}/{build_identifier(run_label, gen, tc, retr)}/iteration-1/ledger-1.csv"

            the_result = spec_check(filename)

            if the_result and total_violations == 1:
                validation_time = 0

            results[gen][tc] = (the_result, validation_time, proposal_sequence)

    
    grand_result[tdir] = results


# print(grand_result)

# Per generation amount of liveness, safety
# Summed up for last x gens

# Per dir -> first generation found, or first generation that found the x'th amount

# Best individual ever found with fitness per generation
# Overall best

entries_proposal_i_v = []
entries_time_i_v = []

entries_proposal_i_c = []
entries_time_i_c = []

viol_config = {}

for tdir, grand in grand_result.items():

    entries_proposal_i = []
    entries_time_i = []
    entries_viol_i = []

    entries_proposal = []
    entries_time = []
    entries_viol = []

    total_violations = 0

    for gen in range(1, 51):
        mean_time_fitness = 0
        mean_proposal_fitness = 0

        for tc in range(1, 11):
            if grand[gen][tc][0]:
                total_violations += 1
                entries_time_i_v.append(grand[gen][tc][1])
                entries_proposal_i_v.append(grand[gen][tc][2])
            else:
                entries_time_i_c.append(grand[gen][tc][1])
                entries_proposal_i_c.append(grand[gen][tc][2])

            entries_time_i.append(grand[gen][tc][1])
            entries_proposal_i.append(grand[gen][tc][2])
            entries_viol_i.append(grand[gen][tc][0])

            mean_time_fitness += grand[gen][tc][1]
            mean_proposal_fitness += grand[gen][tc][2]

        mean_time_fitness /= 10.0
        mean_proposal_fitness /= 10.0

        entries_time.append((gen, mean_time_fitness))
        entries_proposal.append((gen, mean_proposal_fitness))
        entries_viol.append((gen, total_violations))

    print(tdir)
    print("Results Summary:")

    time_plain = [x[1] for x in entries_time]
    print(f"Time fitness")
    print(f"upper whisker={max(time_plain)},")
    print(f"upper quartile={np.percentile(time_plain, 75)},")
    print(f"median={np.percentile(time_plain, 50)},")
    print(f"lower quartile={np.percentile(time_plain, 25)},")
    print(f"lower whisker={min(time_plain)}")

    proposal_plain = [x[1] for x in entries_proposal]
    print(f"Proposal fitness")
    print(f"upper whisker={max(proposal_plain)},")
    print(f"upper quartile={np.percentile(proposal_plain, 75)},")
    print(f"median={np.percentile(proposal_plain, 50)},")
    print(f"lower quartile={np.percentile(proposal_plain, 25)},")
    print(f"lower whisker={min(proposal_plain)}")

    correlation_time, p_value_time = pointbiserialr(entries_viol_i, entries_time_i)
    print(f"Time point biserial p-value: {p_value_time}")
    print(f"Time point biserial coefficient: {correlation_time}")

    correlation_proposal, p_value_proposal = pointbiserialr(entries_viol_i, entries_proposal_i)
    print(f"Proposal point biserial p-value: {p_value_proposal}")
    print(f"Proposal point biserial coefficient: {correlation_proposal}")

    viol_config[tdir] = total_violations

    print(f"Total violations: {total_violations}")

    average_time = sum(entries_time_i_c + entries_time_i_v) / (len(entries_time_i_c) + len(entries_time_i_v))
    print(f"Average f_t: {average_time}")

    stationary_time = adfuller([time[1] for time in entries_time])
    stationary_proposal = adfuller([prop[1] for prop in entries_proposal])

    print(f"Time stationarity p-value: {stationary_time[1]}")
    print(f"Proposal stationarity p-value: {stationary_proposal[1]}")

    print("LaTeX coordinates for violation trajectory:")
    group1_latex = " ".join(f"({x},{y})" for x, y in entries_viol)
    print(group1_latex)
    print("LaTeX coordinates for time fitness trajectory:")
    group2_latex = " ".join(f"({x},{y})" for x, y in entries_time)
    print(group2_latex)
    print("LaTeX coordinates for proposal fitness trajectory:")
    group3_latex = " ".join(f"({x},{y})" for x, y in entries_proposal)
    print(group3_latex)

    print()

print(f"Time fitness (violation)")
print(f"upper whisker={max(entries_time_i_v)},")
print(f"upper quartile={np.percentile(entries_time_i_v, 75)},")
print(f"median={np.percentile(entries_time_i_v, 50)},")
print(f"lower quartile={np.percentile(entries_time_i_v, 25)},")
print(f"lower whisker={min(entries_time_i_v)}")

print(f"Proposal fitness (violation)")
print(f"upper whisker={max(entries_proposal_i_v)},")
print(f"upper quartile={np.percentile(entries_proposal_i_v, 75)},")
print(f"median={np.percentile(entries_proposal_i_v, 50)},")
print(f"lower quartile={np.percentile(entries_proposal_i_v, 25)},")
print(f"lower whisker={min(entries_proposal_i_v)}")

print()

print(f"Time fitness (correct)")
print(f"upper whisker={max(entries_time_i_c)},")
print(f"upper quartile={np.percentile(entries_time_i_c, 75)},")
print(f"median={np.percentile(entries_time_i_c, 50)},")
print(f"lower quartile={np.percentile(entries_time_i_c, 25)},")
print(f"lower whisker={min(entries_time_i_c)}")

print(f"Proposal fitness (correct)")
print(f"upper whisker={max(entries_proposal_i_c)},")
print(f"upper quartile={np.percentile(entries_proposal_i_c, 75)},")
print(f"median={np.percentile(entries_proposal_i_c, 50)},")
print(f"lower quartile={np.percentile(entries_proposal_i_c, 25)},")
print(f"lower whisker={min(entries_proposal_i_c)}")

entries_time_i_complete = entries_time_i_c + entries_time_i_v
entries_proposal_i_complete = entries_proposal_i_c + entries_proposal_i_v
entries_viol_i_complete = [False] * len(entries_time_i_c) + [True] * len(entries_time_i_v)

correlation_time, p_value_time = pointbiserialr(entries_viol_i_complete, entries_time_i_complete)
print(f"Time point biserial p-value: {p_value_time}")
print(f"Time point biserial coefficient: {correlation_time}")

correlation_proposal, p_value_proposal = pointbiserialr(entries_viol_i_complete, entries_proposal_i_complete)
print(f"Proposal point biserial p-value: {p_value_proposal}")
print(f"Proposal point biserial coefficient: {correlation_proposal}")

correlation_fs, p_value_fs = pearsonr(entries_proposal_i_complete, entries_time_i_complete)
print(f"Pearson p-value: {p_value_fs}")
print(f"Pearson r: {correlation_fs}")

print("LaTeX coordinates for proposal and time fitness scatterplot:")
group4_latex = " ".join(f"({x},{y})" for x, y in zip(entries_proposal_i_complete, entries_time_i_complete))
print(group4_latex)

sd_proposal = np.std(entries_proposal_i_complete)
sd_time = np.std(entries_time_i_complete)

residual_scalar = sd_proposal / sd_time * correlation_fs
residual_proposal = [prop - residual_scalar * time for prop, time in zip(entries_proposal_i_complete, entries_time_i_complete)]

correlation_residual, p_value_residual = pointbiserialr(entries_viol_i_complete, residual_proposal)
print(f"Residual proposal point biserial p-value: {p_value_residual}")
print(f"Residual proposal point biserial coefficient: {correlation_residual}")

success_baseline = viol_config["./logs/AMG_Random"]
n = 500

print(viol_config)

for configuration, total_violations in viol_config.items():
    if configuration == "./logs/AMG_Random":
        continue
    table = [
        [total_violations, n - total_violations],
        [success_baseline, n - success_baseline]
    ]

    p_outperforms_baseline = boschloo_exact(table, alternative='greater')

    print(f"{configuration} is better with a p of {p_outperforms_baseline}")


./logs/AMG_Random
Results Summary:
Time fitness
upper whisker=6.945208686385326,
upper quartile=5.4515787708125085,
median=4.749534248229809,
lower quartile=4.349849430159611,
lower whisker=3.5691808163733953
Proposal fitness
upper whisker=1401.3,
upper quartile=1339.575,
median=1306.1999999999998,
lower quartile=1260.2749999999999,
lower whisker=1128.1
Time point biserial p-value: 7.189111111515043e-25
Time point biserial coefficient: 0.4381495365066078
Proposal point biserial p-value: 5.4593224616482095e-05
Proposal point biserial coefficient: -0.17943490761447445
Total violations: 16
Average f_t: 4.9025176234428764
Time stationarity p-value: 8.733767738520834e-08
Proposal stationarity p-value: 2.8738407808873266e-08
LaTeX coordinates for violation trajectory:
(1,0) (2,0) (3,0) (4,0) (5,0) (6,0) (7,0) (8,1) (9,2) (10,2) (11,2) (12,2) (13,3) (14,3) (15,3) (16,3) (17,3) (18,3) (19,3) (20,3) (21,3) (22,3) (23,3) (24,3) (25,5) (26,6) (27,6) (28,6) (29,8) (30,8) (31,9) (32,9) (33,10) (34,